# Pertemuan 10 - Aktivitas Hands-on: Prediksi Customer Churn  
Nama: Novi Shandi  
NIM: 240401010291

In [28]:
import urllib.request
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
urllib.request.urlretrieve(url, "telco_churn.csv")
print("Dataset berhasil diunduh.")

Dataset berhasil diunduh.


Keterangan:  
Kode di atas mengunduh dataset dari internet dan menyimpannya sebagai file telco_churn.csv di dalam notebook. Hasilnya muncul pesan "Dataset berhasil diunduh." yang menandakan file sudah siap dipakai.

In [29]:
import pandas as pd

df = pd.read_csv("telco_churn.csv")
print("Shape:", df.shape)
print("\nProporsi kelas Churn:")
print(df["Churn"].value_counts(normalize=True).round(3))
df.head()

Shape: (7043, 21)

Proporsi kelas Churn:
Churn
No     0.735
Yes    0.265
Name: proportion, dtype: float64


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


Keterangan:  
Kode di atas membuka file dataset dan menghitung proporsi pelanggan yang churn. Hasilnya: dataset berisi 7.043 baris dan 21 kolom, dengan 73,4% pelanggan "No" (tetap) dan 26,6% "Yes" (churn). Karena kelas churn jauh lebih sedikit, dataset ini tergolong imbalanced (tidak seimbang).

In [30]:
# Jebakan 1: kolom TotalCharges terbaca sebagai teks karena ada 11 nilai kosong
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna(subset=["TotalCharges"])

# Jebakan 2: customerID hanya ID unik, tidak berguna untuk prediksi -> buang
df = df.drop(columns=["customerID"])

# Jebakan 3: target Yes/No harus diubah ke 1/0
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# Pisahkan target, lalu encoding semua fitur kategorikal
y = df["Churn"]
X = pd.get_dummies(df.drop(columns=["Churn"]), drop_first=True)

print("Shape X setelah encoding:", X.shape)

Shape X setelah encoding: (7032, 30)


Keterangan:  
Kode di atas membersihkan data: mengubah kolom TotalCharges menjadi angka dan membuang 11 baris yang kosong, menghapus kolom customerID, dan mengubah target Churn dari Yes/No menjadi 1/0. Fitur kategorikal lalu diubah menjadi angka dengan get_dummies. Hasilnya data siap dilatih dengan 7.032 baris dan 30 kolom fitur.

In [31]:
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print("Train:", X_tr.shape, "| Test:", X_te.shape)

Train: (5625, 30) | Test: (1407, 30)


Keterangan:  
Kode di atas membagi data menjadi 80% untuk melatih model dan 20% untuk menguji. Hasilnya: 5.625 baris data latih dan 1.407 baris data uji. Opsi stratify=y menjaga proporsi churn tetap sama di kedua bagian.

In [32]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",   # kunci penanganan imbalance
    random_state=42)
rf.fit(X_tr, y_tr)
print("Model selesai dilatih.")

Model selesai dilatih.


Keterangan:  
Kode di atas membangun dan melatih model Random Forest berisi 300 pohon keputusan. Opsi class_weight="balanced" membuat model memberi perhatian lebih pada kelas churn yang jumlahnya sedikit. Hasilnya muncul pesan "Model selesai dilatih." yang menandakan model siap digunakan.

In [33]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

pred = rf.predict(X_te)
proba = rf.predict_proba(X_te)[:, 1]   # probabilitas kelas churn (1)

print(confusion_matrix(y_te, pred))
print(classification_report(y_te, pred, digits=3))
print("ROC-AUC:", round(roc_auc_score(y_te, proba), 3))

[[925 108]
 [191 183]]
              precision    recall  f1-score   support

           0      0.829     0.895     0.861      1033
           1      0.629     0.489     0.550       374

    accuracy                          0.787      1407
   macro avg      0.729     0.692     0.706      1407
weighted avg      0.776     0.787     0.778      1407

ROC-AUC: 0.82


Keterangan:  
Kode di atas mengukur performa model pada data uji. Hasilnya untuk kelas churn (kelas 1): precision 0,629, recall 0,489, F1-score 0,550, dan ROC-AUC 0,820. Nilai recall 0,489 berarti model berhasil menangkap sekitar 49% pelanggan yang benar-benar akan churn. Akurasi keseluruhan model adalah 0,787 (78,7%).

In [34]:
imp = pd.DataFrame({"Fitur": X.columns,
    "Importance": rf.feature_importances_
    }).sort_values("Importance", ascending=False)
print(imp.head(10))

                             Fitur  Importance
3                     TotalCharges    0.178447
1                           tenure    0.165076
2                   MonthlyCharges    0.151884
25               Contract_Two year    0.059238
10     InternetService_Fiber optic    0.040405
28  PaymentMethod_Electronic check    0.037501
24               Contract_One year    0.029965
13              OnlineSecurity_Yes    0.028844
4                      gender_Male    0.025493
26            PaperlessBilling_Yes    0.023603


Keterangan:  
Kode di atas mengurutkan fitur berdasarkan pengaruhnya terhadap prediksi. Hasilnya tiga fitur paling berpengaruh adalah TotalCharges (total tagihan), tenure (lama berlangganan), dan MonthlyCharges (tagihan bulanan). Ini menunjukkan bahwa lama berlangganan dan besarnya tagihan adalah faktor utama penentu churn.

In [35]:
hasil = pd.DataFrame({
    "Prob_Churn": proba,
    "Aktual": y_te.values
}).sort_values("Prob_Churn", ascending=False)
print(hasil.head(10))   # 10 pelanggan paling berisiko churn

      Prob_Churn  Aktual
369     1.000000       1
1220    0.996667       1
728     0.996667       0
107     0.996667       0
304     0.996667       1
31      0.990000       1
261     0.983333       1
591     0.970000       1
1149    0.963333       1
446     0.960000       0


Keterangan:  
Kode di atas menghitung probabilitas churn untuk tiap pelanggan lalu menampilkan 10 pelanggan paling berisiko. Hasilnya menunjukkan pelanggan-pelanggan dengan probabilitas churn mendekati 1,0 (di atas 0,96), yang berarti sangat berisiko berhenti berlangganan. Daftar ini dapat dipakai perusahaan untuk memprioritaskan tindakan pencegahan.

## Kesimpulan  
1. Dataset Telco Customer Churn bersifat imbalanced, dengan hanya 26,6% pelanggan yang churn, sehingga Accuracy saja menyesatkan dan Recall kelas churn menjadi metrik yang lebih penting.  
2. Model Random Forest dengan class_weight="balanced" menghasilkan akurasi 78,7%, ROC-AUC 0,820, dan Recall kelas churn 0,489 (menangkap sekitar 49% pelanggan yang benar-benar churn).  
3. Fitur paling berpengaruh terhadap churn adalah TotalCharges, tenure, dan MonthlyCharges, sehingga lama berlangganan dan besarnya tagihan menjadi faktor utama yang perlu diperhatikan perusahaan.  
4. Model ini membantu perusahaan mengidentifikasi pelanggan berisiko tinggi lebih dini, sehingga tim retensi dapat memprioritaskan tindakan pencegahan sebelum pelanggan benar-benar berhenti berlangganan.